# 02 — Profile Bronze & Auto-Generate DQX Rules

Two DQX features in one notebook:

1. **`DQProfiler`** — scans the bronze pitches table and returns summary statistics + per-column **profiles** (nulls, ranges, distinct values).
2. **`DQGenerator`** — turns those profiles into candidate DQX rules.

We dump the generated rules to `checks.generated.yml` so you can diff them against
the hand-curated `checks.yml`. In production you'd review the auto-generated set,
tighten thresholds, and commit a final version.

> Profiling is a **one-time discovery step** — the DQX docs explicitly call out
> that you don't run this on a schedule.

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.engine import DQEngine
from dotenv import load_dotenv
import os, yaml, json
from pathlib import Path

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
ws    = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "dqx_baseball")
BRONZE     = f"{UC_CATALOG}.{UC_SCHEMA}_bronze.pitches_raw"

print(f"Profiling: {BRONZE}")

Profiling: alexander_booth.dqx_baseball_bronze.pitches_raw


In [2]:
# Pull bronze and let DQProfiler do its thing.
# Default sample is 30% of rows, capped at 1,000 — fine for a discovery pass.
df = spark.read.table(BRONZE)

profiler = DQProfiler(ws)
summary_stats, profiles = profiler.profile(df)

print(f"Got {len(profiles)} column profiles")
print("\nSummary stats:")
print(json.dumps(summary_stats, indent=2, default=str)[:1500])

Got 13 column profiles

Summary stats:
{
  "at_bat_id": {
    "count": 1000,
    "mean": null,
    "stddev": null,
    "min": null,
    "25%": null,
    "50%": null,
    "75%": null,
    "max": null,
    "count_null": 0,
    "count_non_null": 1000,
    "count_distinct": 1000,
    "empty_count": 0
  },
  "game_id": {
    "count": 1000,
    "mean": null,
    "stddev": null,
    "min": null,
    "25%": null,
    "50%": null,
    "75%": null,
    "max": null,
    "count_null": 0,
    "count_non_null": 1000,
    "count_distinct": 570,
    "empty_count": 0
  },
  "inning": {
    "count": 1000,
    "mean": 6.013,
    "stddev": 6.852721414940973,
    "min": 1,
    "25%": 3,
    "50%": 5,
    "75%": 8,
    "max": 27,
    "count_null": 0,
    "count_non_null": 1000,
    "count_distinct": 22,
    "empty_count": 0
  },
  "pitcher_id": {
    "count": 1000,
    "mean": null,
    "stddev": null,
    "min": null,
    "25%": null,
    "50%": null,
    "75%": null,
    "max": null,
    "count_null": 14,

In [3]:
# Inspect a couple of profile entries — each DQProfile is one rule *candidate*
# (so you'll see multiple per column: e.g. is_not_null + min_max for pitch_speed_mph).
from collections import Counter
by_col = Counter(p.column for p in profiles)
for col, n in by_col.items():
    print(f"  {col:<18} {n} candidate rule(s)")

print("\nFirst few raw DQProfile entries:")
for p in profiles[:6]:
    print(f"  name={p.name!r:<24} column={p.column!r:<18} parameters={p.parameters}")

  at_bat_id          1 candidate rule(s)
  game_id            1 candidate rule(s)
  game_date          2 candidate rule(s)
  inning             2 candidate rule(s)
  pitcher_id         1 candidate rule(s)
  batter_id          1 candidate rule(s)
  pitch_type         1 candidate rule(s)
  pitch_speed_mph    2 candidate rule(s)
  batting_avg        2 candidate rule(s)

First few raw DQProfile entries:
  name='is_not_null_or_empty'   column='at_bat_id'        parameters={'trim_strings': True}
  name='is_not_null_or_empty'   column='game_id'          parameters={'trim_strings': True}
  name='is_not_null'            column='game_date'        parameters=None
  name='min_max'                column='game_date'        parameters={'min': datetime.date(2026, 3, 28), 'max': datetime.date(2026, 7, 2)}
  name='is_not_null'            column='inning'           parameters=None
  name='min_max'                column='inning'           parameters={'min': 1, 'max': 27}


In [4]:
# Convert profiles → DQX checks.
# By default DQGenerator emits criticality='error'; we override to 'warn' so the
# auto-generated rules don't compete with the hand-tuned error-level ones in checks.yml.
generator   = DQGenerator(ws)
auto_checks = generator.generate_dq_rules(profiles, criticality="warn")

print(f"Generated {len(auto_checks)} candidate checks\n")
for c in auto_checks[:6]:
    print(json.dumps(c, indent=2, default=str))
    print("---")

09:46:28     INFO [d.l.d.profiler.generator] ✅ Quality rules generation completed. Generated 13 rules.


Generated 13 candidate checks

{
  "check": {
    "function": "is_not_null_and_not_empty",
    "arguments": {
      "column": "at_bat_id",
      "trim_strings": true
    }
  },
  "name": "at_bat_id_is_null_or_empty",
  "criticality": "warn"
}
---
{
  "check": {
    "function": "is_not_null_and_not_empty",
    "arguments": {
      "column": "game_id",
      "trim_strings": true
    }
  },
  "name": "game_id_is_null_or_empty",
  "criticality": "warn"
}
---
{
  "check": {
    "function": "is_not_null",
    "arguments": {
      "column": "game_date"
    }
  },
  "name": "game_date_is_null",
  "criticality": "warn"
}
---
{
  "check": {
    "function": "is_in_range",
    "arguments": {
      "column": "game_date",
      "min_limit": "2026-03-28",
      "max_limit": "2026-07-02"
    }
  },
  "name": "game_date_isnt_in_range",
  "criticality": "warn"
}
---
{
  "check": {
    "function": "is_not_null",
    "arguments": {
      "column": "inning"
    }
  },
  "name": "inning_is_null",
  "critica

In [5]:
# Validate the generated checks before committing them — DQX has a built-in linter.
status = DQEngine.validate_checks(auto_checks)
print(f"Validation: {status}")
assert not status.has_errors, status

Validation: No errors found


In [6]:
# Persist alongside the hand-written checks.yml so they can be diffed easily.
out_path = Path.cwd() / "checks.generated.yml"
out_path.write_text(yaml.safe_dump(auto_checks, sort_keys=False))
print(f"Wrote {out_path}  ({out_path.stat().st_size:,} bytes)")

Wrote /Users/alexander.booth/Documents/Repos/databricks_demos/dqx-baseball-data-quality/checks.generated.yml  (2,042 bytes)


## What to notice

- `pitch_speed_mph` profile picks up our **dirty** mins/maxes (negative speeds, 140 mph) —
  reason DQX docs say the auto-generated rules are *candidates*, not production rules.
  Hand-tuned `checks.yml` uses `40–110`.
- `batting_avg` similarly pulls bogus min/max from polluted rows.
- The profiler typically emits `is_not_null` and range/in-list candidates — exactly
  what you'd want to seed a real review.

Continue with `03_apply_checks_and_quarantine`.